In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import yaml
import pathlib
from pathlib import Path
import shutil

In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# pdf
#--------------------------------------------------------------------------------------------------------------------------

data_name = "Default"
if_diag_only = False

#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

before_dir = Path(f"{data_name}/Cutted")

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

after_dir = Path(f"{data_name}/Covariance_Matrices")

Auxilary Functions

In [3]:
def Build_Uncorrelated_Matrix(df, prefix="error_unc_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    cols_squared_sum = (df[cols].pow(2).sum(axis=1))
    return (np.diag(cols_squared_sum))

def Build_Additive_Matrix(df, prefix="error_add_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    Matrix_total = np.zeros((len(df), len(df)))
    for i in range(len(cols)):
        col = df[cols[i]]
        Matrix = np.outer(col, col)
        Matrix_total = Matrix_total + Matrix 
    return Matrix_total

def Build_Multiplicative_Matrix(df, prefix="error_mult_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    Matrix_total = np.zeros((len(df), len(df)))
    for i in range(len(cols)):
        col_xsec = df["xsec"]
        col = df[cols[i]]*col_xsec
        Matrix = np.outer(col, col)
        Matrix_total = Matrix_total + Matrix 
    return Matrix_total

def Build_Covariance_Matrix(df):
    Matrix_uncorrelated = Build_Uncorrelated_Matrix(df)
    Matrix_additive = Build_Additive_Matrix(df)
    Matrix_multiplicative = Build_Multiplicative_Matrix(df)
    Matrix_total = Matrix_uncorrelated + Matrix_additive + Matrix_multiplicative

    if if_diag_only:
        Matrix_total = np.diag(np.diag(Matrix_total))

    return Matrix_total

Test

In [4]:
np.set_printoptions(precision=9, suppress=True, linewidth=300)
df = pd.read_csv(before_dir / "DY/E605/E605-7Q8.csv")
display(Build_Uncorrelated_Matrix(df))
display(Build_Additive_Matrix(df))
display(Build_Multiplicative_Matrix(df))
display(Build_Covariance_Matrix(df))

array([[0.030171233, 0.         , 0.         , 0.         , 0.         , 0.         , 0.         ],
       [0.         , 0.03344816 , 0.         , 0.         , 0.         , 0.         , 0.         ],
       [0.         , 0.         , 0.022511645, 0.         , 0.         , 0.         , 0.         ],
       [0.         , 0.         , 0.         , 0.021069712, 0.         , 0.         , 0.         ],
       [0.         , 0.         , 0.         , 0.         , 0.010684197, 0.         , 0.         ],
       [0.         , 0.         , 0.         , 0.         , 0.         , 0.009928285, 0.         ],
       [0.         , 0.         , 0.         , 0.         , 0.         , 0.         , 0.005660923]])

array([[0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.]])

array([[0.013865062, 0.017185612, 0.014571562, 0.014218312, 0.010049962, 0.009484762, 0.00685305 ],
       [0.017185612, 0.021301403, 0.018061312, 0.017623462, 0.012456832, 0.011756272, 0.00849429 ],
       [0.014571562, 0.018061312, 0.015314062, 0.014942812, 0.010562062, 0.009968062, 0.00720225 ],
       [0.014218312, 0.017623462, 0.014942812, 0.014580562, 0.010306012, 0.009726412, 0.00702765 ],
       [0.010049962, 0.012456832, 0.010562062, 0.010306012, 0.007284622, 0.006874942, 0.00496737 ],
       [0.009484762, 0.011756272, 0.009968062, 0.009726412, 0.006874942, 0.006488302, 0.00468801 ],
       [0.00685305 , 0.00849429 , 0.00720225 , 0.00702765 , 0.00496737 , 0.00468801 , 0.00338724 ]])

array([[0.044036295, 0.017185612, 0.014571562, 0.014218312, 0.010049962, 0.009484762, 0.00685305 ],
       [0.017185612, 0.054749562, 0.018061312, 0.017623462, 0.012456832, 0.011756272, 0.00849429 ],
       [0.014571562, 0.018061312, 0.037825707, 0.014942812, 0.010562062, 0.009968062, 0.00720225 ],
       [0.014218312, 0.017623462, 0.014942812, 0.035650274, 0.010306012, 0.009726412, 0.00702765 ],
       [0.010049962, 0.012456832, 0.010562062, 0.010306012, 0.017968819, 0.006874942, 0.00496737 ],
       [0.009484762, 0.011756272, 0.009968062, 0.009726412, 0.006874942, 0.016416587, 0.00468801 ],
       [0.00685305 , 0.00849429 , 0.00720225 , 0.00702765 , 0.00496737 , 0.00468801 , 0.009048163]])

DY

In [5]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]
excludes = []

In [6]:
processes = ["DY","SIDIS"]

for process in processes:
    process_dir = after_dir / process
    if process_dir.exists():
        shutil.rmtree(process_dir) 
    process_dir.mkdir(parents=True, exist_ok=True)

In [7]:
for experiment in experiments:

    exp_dir = before_dir / "DY" / experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]
    file_paths = [exp_dir / f"{name}.csv" for name in file_names] 

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        file_path = exp_dir / f"{file_name}.csv"

        df = pd.read_csv(file_path)
        
        Matrix = Build_Covariance_Matrix(df)

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        if destination.exists():
            raise FileExistsError(f"{file_name} already exists")
        pd.DataFrame(Matrix).to_csv(destination, index=False)
        
        print(f"{file_name} generated")

ATLAS7-00y10 generated
ATLAS7-10y20 generated
ATLAS7-20y24 generated
ATLAS8-00y04 generated
ATLAS8-04y08 generated
ATLAS8-08y12 generated
ATLAS8-116Q150 generated
ATLAS8-12y16 generated
ATLAS8-16y20 generated
ATLAS8-20y24 generated
ATLAS8-46Q66 generated
ATLAS13 generated
CDF1 generated
CDF2 generated
CMS7 generated
CMS8 generated
CMS13-00y04 generated
CMS13-04y08 generated
CMS13-08y12 generated
CMS13-106Q170 generated
CMS13-12y16 generated
CMS13-16y24 generated
CMS13-170Q350 generated
CMS13-350Q1000 generated
D01 generated
D02 generated
D02mu generated
E228-200-4Q5 generated
E228-200-5Q6 generated
E228-200-6Q7 generated
E228-200-7Q8 generated
E228-200-8Q9 generated
E228-300-11Q12 generated
E228-300-4Q5 generated
E228-300-5Q6 generated
E228-300-6Q7 generated
E228-300-7Q8 generated
E228-300-8Q9 generated
E228-400-11Q12 generated
E228-400-12Q13 generated
E228-400-13Q14 generated
E228-400-5Q6 generated
E228-400-6Q7 generated
E228-400-7Q8 generated
E228-400-8Q9 generated
E605-10.5Q11.5 gen